In [1]:
import pandas as pd
from geopy.distance import geodesic
from scipy.spatial import cKDTree
import numpy as np

In [39]:
faults = pd.read_csv("../data/J1939Faults.csv")
faults.head(10)

/var/folders/h8/51mqc9v517x00_mhcb5s7ncw0000gn/T/ipykernel_15672/2360368126.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  faults = pd.read_csv("../data/J1939Faults.csv")


,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
0,1,990349,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,111,17,True,2,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:34:25.000
1,2,990360,2015-02-21 11:34:34.000,NaN,NaN,unknown,unknown,unknown,unknown,11,629,12,True,127,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:35:10.000
2,3,990364,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,1807,2,False,127,NaN,1369,105336226,41.421250,-87.767361,2015-02-21 11:35:26.000
3,4,990370,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,1807,2,True,127,NaN,1369,105336226,41.421018,-87.767361,2015-02-21 11:36:08.000
4,5,990416,2015-02-21 11:39:41.000,NaN,NaN,22281684P01*22357957P01*22362082P01*,13063430,0USA13_13_0415_2238A,VOLVO,0,4364,17,False,2,NaN,1674,105427130,38.416481,-89.442638,2015-02-21 11:39:37.000
5,6,990431,2015-02-21 11:40:22.000,Low (Severity Low) Engine Coolant Level,NaN,04993120*00025921*082113134117*07700053*I0*BBZ*,79466580,6X1u10D1500000000,CMMNS,0,111,17,True,1,NaN,1417,105438630,33.043564,-96.179722,2015-02-21 11:40:59.000
6,7,990439,2015-02-21 11:40:52.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,111,17,True,2,NaN,1597,105344243,36.902916,-86.436481,2015-02-21 11:41:29.000
7,8,990441,2015-02-21 11:40:22.000,Low (Severity Low) Engine Coolant Level,NaN,04993120*00022630*082113134117*07700053*I0*BBZ*,79463845,6X1u10D1500000000,CMMNS,0,111,17,True,1,NaN,1429,105356054,38.228796,-84.582500,2015-02-21 11:41:44.000
8,9,990442,2015-02-21 11:40:22.000,High (Severity Low) Water In Fuel Indicator,NaN,04993120*00022630*082113134117*07700053*I0*BBZ*,79463845,6X1u10D1500000000,CMMNS,0,97,15,True,1,NaN,1429,105356054,38.228796,-84.582500,2015-02-21 11:41:44.000
9,10,990446,2015-02-21 11:41:55.000,Low (Severity Low) Engine Coolant Level,NaN,04993120*00025921*082113134117*07700053*I0*BBZ*,79466580,6X1u10D1500000000,CMMNS,0,111,17,False,1,NaN,1417,105438630,33.039953,-96.182592,2015-02-21 11:41:51.000


### Possible Features:
* RecordID is unique
* EventTimeStamp has 1,050,909 unique values (no nan)
* eventDescription has 60,845 nan 
* ecuSoftwareVersion has 1,899 unique values (296,050 nan)
* ecuModel has 30 unique values (64,758 nan)
* ecuMake has 23 unique values (64,758 nan)
* ecuSource has 5 unique values (no nan)
* spn has 450 unique values (no nan)
* fmi has 26 unique values (no nan)
* active has 2 unique values (no nan)
* EquipmentID has 1,927 unique values (no nan)
* MCTNumber has 768 unique values (no nan)
* Latitude has 211,823 unique values (no nan)
* Longitude has 265,211 unique values (no nan)
* LocationTimeStamp has 1,036,006 unique values (no nan)

### Drop Columns:
* ESS_Id
* actionDescription
* ecuSerialNumber
* faultValue

In [3]:
len(faults['LocationTimeStamp'].unique())

1036006

In [4]:
faults['LocationTimeStamp'].isna().sum()

np.int64(0)

In [5]:
# Convert timestamps to datetime objects
faults['EventTimeStamp'] = pd.to_datetime(faults['EventTimeStamp'])
faults['LocationTimeStamp'] = pd.to_datetime(faults['LocationTimeStamp'])

In [6]:
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")
diagnostics.head(2)

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1


### To get the on-board diagnostics at the time of the fault code, we can match the **RecordID** to the **FaultId**.

In [7]:
diagnostics.loc[diagnostics['FaultId'] == 1]

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
5,6,AcceleratorPedal,0,1
6,7,IntakeManifoldTemperature,78.8,1
7,8,FuelRate,0,1
8,9,FuelLtd,12300.907429328,1
9,10,EngineRpm,0,1


This data is in long-format, so each FaultId can have potentially many diagnostic values.

**Note:** Not all diagnostic values are recorded for all faults, so you will have a large number of missing values.

For example, for the second fault code in our dataset, only the ignition status and lamp status were recorded.

In [8]:
diagnostics.loc[diagnostics['FaultId'] == 46]

,Id,Name,Value,FaultId
418,419,IgnStatus,True,46
419,420,LampStatus,22527,46


Finally, we can get a little bit more information about the different fault codes from the Service Fault Codes spreadsheet.

In [9]:
sfc = pd.read_excel("../data/Service Fault Codes_1_0_0_167.xlsx")
sfc.head(2)

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
1,Y,112,167,Not Mapped,20,128,7,635,7,Not Mapped,Red,Stop / Shutdown,Engine Timing Actuator Driver Circuit - Mechan...,Mechanical failure in the engine timing actuat...


For a large number of fault codes, there are multiple records. For example, if we look at the rows for the first fault in our dataset, we see that there are two rows.

In [10]:
(
    sfc
    .loc[sfc['SPN'] == 5246]
    .loc[sfc['J1939 FMI'].isin([ 0, 15, 16, 19, 14])]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
2518,Y,3712,167,Not Mapped,Not Mapped,Not Mapped,0,5246,0,Not Mapped,Red,Stop / Shutdown,Aftertreatment SCR Operator Inducement - Data ...,SCR inducement of 5 mph derate - Fault Code 41...
2781,Y,4134,167,Not Mapped,Not Mapped,Not Mapped,0,5246,15,Not Mapped,Amber,Warning,Aftertreatment SCR Operator Inducement - Data ...,SCR inducement - Least Severe - Fault Code 371...
4338,Y,6254,167,Not Mapped,Not Mapped,Not Mapped,0,5246,16,Not Mapped,Amber,Warning,Aftertreatment SCR Operator Inducement Severit...,NaN


Or even more.

In [11]:
(
    sfc
    .loc[sfc['SPN'] == 629]
    .loc[sfc['J1939 FMI'] == 12]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
180,Y,343,167,Not Mapped,254,0,12,629,12,P0607,Amber,Warning,Engine Control Module Warning Internal Hardwar...,ECM power supply errors have been detected.
689,Y,1116,167,Not Mapped,254,0,12,629,12,Not Mapped,Amber,Warning,Engine Control Module Critical Internal Failur...,ECM Internal failure has occurred.
854,Y,1388,167,Not Mapped,254,0,12,629,12,Not Mapped,NaN,NaN,Engine Control Module Data Lost - Bad Intellig...,The ECM data has been lost.
1019,Y,1597,167,Not Mapped,254,0,12,629,12,Not Mapped,Maintenance,Maintenance,Engine Control Module Critical Internal Failur...,The ECM has occurred an internal failure.


In [12]:
faults[faults['spn'] == 5246]

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
45,46,990931,2015-02-21 12:10:51,NaN,NaN,04993120*00027849*082113134117*07700053*I0*BBZ*,79464664,6X1u10D1500000000,CMMNS,0,5246,0,True,1,NaN,1395,105349612,36.065972,-86.433425,2015-02-21 12:11:27
1918,1919,1007751,2015-02-22 19:44:55,NaN,NaN,04993120*00027849*082113134117*07700053*I0*BBZ*,79464664,6X1u10D1500000000,CMMNS,0,5246,0,True,1,NaN,1395,105349612,36.066203,-86.434814,2015-02-22 19:46:27
2058,2059,1010486,2015-02-23 04:00:21,NaN,NaN,04993120*00027849*082113134117*07700053*I0*BBZ*,79464664,6X1u10D1500000000,CMMNS,0,5246,0,False,1,NaN,1395,105349612,36.066666,-86.434537,2015-02-23 01:06:06
2089,2090,1011009,2015-02-23 05:05:44,NaN,NaN,05290170*03015749*051914190353*09400015*G1*BDR*,79642446,6X1u13D1500000000,CMMNS,0,5246,0,True,1,NaN,1630,105329900,40.733009,-74.087777,2015-02-23 05:08:23
2971,2972,1026305,2015-02-23 15:54:22,NaN,NaN,unknown,unknown,unknown,unknown,0,5246,0,True,1,NaN,1487,105369355,28.077361,-81.897083,2015-02-23 15:54:58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183032,1244156,121610128,2020-02-19 07:02:33,NaN,NaN,05317106*05005224*051718172255*09401583*G1*BDR*,79845785,6X1u13D1500000000,CMMNS,0,5246,0,True,1,NaN,1814,105369518,36.067037,-86.434120,2020-02-19 07:03:09
1183684,1244808,121909497,2020-02-21 07:23:44,NaN,NaN,04384413*22246857*090619141107*60701756*G1*BGT*,80092582,6X1u17D1500000000,CMMNS,0,5246,16,True,1,NaN,2211,105329862,36.066296,-86.434305,2020-02-21 07:24:20
1184328,1245452,122305094,2020-02-24 15:28:05,NaN,NaN,04384413*22246857*090619141107*60701756*G1*BGT*,80092582,6X1u17D1500000000,CMMNS,0,5246,16,False,1,NaN,2211,105329862,36.066620,-86.434722,2020-02-24 15:28:01
1184330,1245454,122305096,2020-02-24 15:27:26,NaN,NaN,04384413*22246857*090619141107*60701756*G1*BGT*,80092582,6X1u17D1500000000,CMMNS,0,5246,0,True,1,NaN,2211,105329862,36.066620,-86.434722,2020-02-24 15:28:02


I know the readme says to remove the records near service locations, however, I think that can be done later. I don't think we need to remove the nan values just yet bc that could remove observations that could be useful.

I think the priority would be converting the timestamps and ordering the records chronologically, then linking the faults RecordIds with diagnostics FaultId. Maybe drop the id column and widen the diagnostics data.

* spn code 5246 has 1195 records with 5 unique fmi codes (no nan)

So, 1,195 records had full derates out of 1,187,335 records. Pretty imbalanced.

## Data Cleaning

### order the data chronologically

In [13]:
faults = faults.sort_values(by='EventTimeStamp', ascending=True)
faults.head(2)

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
1154193,1211417,108604425,2000-03-18 19:14:10,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,829,3,True,126,NaN,2015,105427130,36.935972,-86.507407,2000-03-18 19:14:46
1154194,1211418,108604426,2000-03-18 19:14:10,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,96,3,True,126,NaN,2015,105427130,36.935972,-86.507407,2000-03-18 19:14:46


## service station

In [14]:
service_stations = [
    (36.0666667, -86.4347222),
    (35.5883333, -86.4438888),
    (36.1950, -83.174722)
]

In [15]:
from sklearn.neighbors import BallTree

def is_near_service_station_balltree(df, service_stations, threshold_distance=1.0):
    earth_radius_km = 6371.0
    
    stations_rad = np.radians(service_stations)
    points_rad = np.radians(df[['Latitude', 'Longitude']].values)
    
    tree = BallTree(stations_rad, metric='haversine')
    
    # Query radius in radians
    indices = tree.query_radius(points_rad, r=threshold_distance / earth_radius_km)
    
    is_near = np.array([len(idx) > 0 for idx in indices])
    
    return is_near

In [16]:
threshold_distance = 1.0 

# apply function
faults['IsServiceStation'] = is_near_service_station_balltree(
    faults, service_stations, threshold_distance)

In [17]:
faults['IsServiceStation'].value_counts(normalize = True)

IsServiceStation
False    0.88936
True     0.11064
Name: proportion, dtype: float64

### diagnostics features

In [18]:
diagnostics['Value'] = diagnostics['Value'].replace({'FALSE': 0, 'TRUE': 1})

In [19]:
diagnostics_wide = diagnostics.pivot(index='FaultId', columns='Name', values='Value')
features = diagnostics_wide.reset_index()
features.columns.name = None

## merge faults and feature

In [20]:
combined = pd.merge(faults, 
                    features, 
                    left_on='RecordID', 
                    right_on='FaultId',
                    how = 'left')

## what is a full derate?

In [21]:
combined['IsDerateFull'] = (combined['spn'] == 5246) & (combined['active'] == True)
combined['IsDerateFull'].value_counts()

IsDerateFull
False    1186728
True         607
Name: count, dtype: int64

In [22]:
faults['IsFullDerate'] = (faults['spn'] == 5246) & (faults['active'] == True)

In [23]:
combined_filtered = combined[combined['IsServiceStation'] == False]
combined_filtered = combined_filtered[~((combined_filtered['spn'] == 5246) & (combined_filtered['active'] == False))]
combined_filtered['IsServiceStation'].value_counts(normalize = True)

IsServiceStation
False    1.0
Name: proportion, dtype: float64

### Derate vehicles with highest n faults

In [34]:
all_derate_vehicles = combined_filtered[(combined_filtered['IsDerateFull'] == True)]
grouped_derate_vehicles = all_derate_vehicles.groupby('EquipmentID').size().reset_index(name='row_count')
grouped_derate_vehicles = grouped_derate_vehicles.sort_values(by='row_count', ascending=False)
print('EquipmentID 1524 has the most rows to inspect')
grouped_derate_vehicles

EquipmentID 1524 has the most rows to inspect


,EquipmentID,row_count
38,1524,31
43,1535,23
39,1525,15
45,1539,14
3,305,13
...,...,...
79,1602,1
76,1599,1
74,1595,1
73,1592,1


### severity level in event description

In [25]:
import re

In [26]:
def extract_severity(text):

    if pd.isna(text):
        return np.nan

    #severity with "Low", "Medium", or "high"
    pattern = r'Severity\s+(Low|Medium|High)'

    #find pattern
    match = re.search(pattern, text)

    if match: 
        return f"Severity {match.group(1)}"
    else: 
        return np.nan

combined_filtered['Severity_Level'] = combined_filtered['eventDescription'].apply(extract_severity)


In [27]:
severity_map = {
    'Severity Low': 1,
    'Severity Medium': 2,
    'Severity High': 3
}

combined_filtered['Severity_Level_Numeric'] = combined_filtered['Severity_Level'].map(severity_map)

combined_filtered.loc[combined_filtered['spn'] == 1569, 'Severity_Level_Numeric'] = 4

In [33]:
inspect_column = combined_filtered[['eventDescription', 'Severity_Level','Severity_Level_Numeric']].dropna(subset=['Severity_Level'])
inspect_column

,eventDescription,Severity_Level,Severity_Level_Numeric
6,Low (Severity Medium) Transmission Air Tank Pr...,Severity Medium,2.0
7,Low (Severity Medium) Transmission Air Tank Pr...,Severity Medium,2.0
16,Low (Severity Medium) Engine Coolant Level,Severity Medium,2.0
18,Low (Severity Medium) Transmission Air Tank Pr...,Severity Medium,2.0
19,Low (Severity High) Transmission Air Tank Pres...,Severity High,3.0
...,...,...,...
1187325,Low (Severity Medium) Engine Coolant Level,Severity Medium,2.0
1187326,Low (Severity Medium) Engine Coolant Level,Severity Medium,2.0
1187327,Low (Severity Low) Catalyst Tank Level,Severity Low,1.0
1187330,Low (Severity Medium) Engine Coolant Level,Severity Medium,2.0


In [35]:
combined_filtered

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,...,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure,IsDerateFull,Severity_Level,Severity_Level_Numeric
0,1211417,108604425,2000-03-18 19:14:10,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,1279,False,NaN,0,NaN,100,0.58,False,NaN,NaN
1,1211418,108604426,2000-03-18 19:14:10,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,1279,False,NaN,0,NaN,100,0.58,False,NaN,NaN
2,1211419,108604487,2000-03-18 19:20:47,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,255,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
3,1211420,108604488,2000-03-18 19:20:47,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,255,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
4,1211422,108608408,2000-03-19 02:59:58,Not Reporting Data Wheel Sensor ABS Axle 2 Right,NaN,AAAI000032*AAAM000038*BB41275 *A82J140721A_9...,5W26153559,EC80ESP,BNDWS,11,...,1279,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187327,1248454,123904424,2020-03-06 14:00:26,Low (Severity Low) Catalyst Tank Level,NaN,04384413*22383729*082218154102*60701732*G1*BGT*,80156139,6X1u17D1500000000,CMMNS,0,...,1023,NaN,NaN,NaN,NaN,NaN,NaN,False,Severity Low,1.0
1187328,1248455,123905139,2020-03-06 14:04:23,Condition Exists Engine Protection Torque Derate,NaN,04358814*06099720*030816202706*09400153*G1*BDR*,79932020,6X1u13D1500000000,CMMNS,0,...,18431,False,NaN,65.01096,NaN,73.2,7.83,False,NaN,4.0
1187329,1248456,123905996,2020-03-06 14:13:38,Abnormal Rate of Change Aftertreatment 1 Intak...,NaN,05317106*05100987*050719120655*09401585*G1*BDR*,79880653,6X1u13D1500000000,CMMNS,0,...,17407,NaN,NaN,66.5741,NaN,100,6.96,False,NaN,NaN
1187330,1248457,123906113,2020-03-06 14:14:13,Low (Severity Medium) Engine Coolant Level,NaN,04384413*22544852*090619141107*60701756*G1*BGT*,NaN,NaN,NaN,0,...,1023,False,NaN,11.84489,14.1,100,1.74,False,Severity Medium,2.0


## convert features to numeric

In [40]:
feature_numeric_columns = [
    'AcceleratorPedal', 'BarometricPressure', 'CruiseControlSetSpeed', 
    'DistanceLtd', 'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure', 'EngineOilTemperature',
    'EngineRpm', 'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature', 
    'IntakeManifoldTemperature', 'ParkingBrake', 'ServiceDistance', 'Speed', 
    'SwitchedBatteryVoltage', 'Throttle', 'TurboBoostPressure'
]

combined_filtered[feature_numeric_columns] = combined_filtered[feature_numeric_columns].apply(pd.to_numeric, errors='coerce')

now i wanted to make sure that all features were numeric before modeling

### false derates or if there was any subsequent derates within 24 hour derates

In [42]:
inspect_derate_rows = combined_filtered[combined_filtered['IsDerateFull'] == True][['IsDerateFull', 'active']].value_counts()
inspect_derate_rows

IsDerateFull  active
True          True      498
Name: count, dtype: int64

In [43]:
combined_filtered['EventTimeStamp'] = pd.to_datetime(combined_filtered['EventTimeStamp'])
combined_filtered['IsDerateActual'] = (combined_filtered['spn'] == 5246)

In [46]:
#grabbing only derate events with equipment ID and timestamp
derate_events = combined_filtered[combined_filtered['spn'] == 5246].copy(0)
derate_events = derate_events.sort_values(['EquipmentID', 'EventTimeStamp'])

In [49]:
duplicate_indices = []

for equipment_id, group in derate_events.groupby('EquipmentID'):
    group = group.reset_index()

    last_valid_timestamp = None

    for i, row in group.iterrows():
        current_timestamp = row['EventTimeStamp']

        if last_valid_timestamp is None:
            last_valid_timestamp = current_timestamp
        elif (current_timestamp - last_valid_timestamp).total_seconds() < 24 * 3600:
            duplicate_indices.append(row['index'])
        else:
            last_valid_timestamp = current_timestamp
if duplicate_indices:
    combined_filtered.loc[duplicate_indices, 'IsDerateActual'] = False

In [54]:
verification_df = combined_filtered[combined_filtered['spn'] == 5246][['EventTimeStamp', 'EquipmentID', 'spn', 'IsDerateActual']].sort_values(['EquipmentID', 'EventTimeStamp'])
verification_df['TimeSincePrevDerate'] = verification_df.groupby('EquipmentID')['EventTimeStamp'].diff()
verification_df

,EventTimeStamp,EquipmentID,spn,IsDerateActual,TimeSincePrevDerate
516702,2016-07-12 19:11:07,301,5246,True,NaT
1171274,2020-01-06 10:13:57,302,5246,True,NaT
1173048,2020-01-13 13:18:31,302,5246,True,7 days 03:04:34
1182001,2020-02-14 11:21:54,302,5246,True,31 days 22:03:23
377013,2016-02-15 10:59:28,304,5246,True,NaT
...,...,...,...,...,...
998450,2018-07-10 13:37:00,1942,5246,False,0 days 02:30:04
1006602,2018-08-06 10:37:53,1946,5246,True,NaT
990700,2018-06-14 14:43:49,1978,5246,True,NaT
993984,2018-06-25 14:48:16,305,5246,True,NaT


In [55]:
combined_filtered['IsDerateActual'].value_counts()

IsDerateActual
False    1055175
True         355
Name: count, dtype: int64

filter out false derates

In [56]:
combined_filtered

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,...,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure,IsDerateFull,Severity_Level,Severity_Level_Numeric,IsDerateActual
0,1211417,108604425,2000-03-18 19:14:10,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,NaN,NaN,0.00000,NaN,100.0,0.58,False,NaN,NaN,False
1,1211418,108604426,2000-03-18 19:14:10,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,NaN,NaN,0.00000,NaN,100.0,0.58,False,NaN,NaN,False
2,1211419,108604487,2000-03-18 19:20:47,High Voltage (Fuel Level),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,False
3,1211420,108604488,2000-03-18 19:20:47,High Voltage (Left Fuel Level Sensor),NaN,NaN,NaN,CECU3B-NAMUX4,PACCR,49,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,False
4,1211422,108608408,2000-03-19 02:59:58,Not Reporting Data Wheel Sensor ABS Axle 2 Right,NaN,AAAI000032*AAAM000038*BB41275 *A82J140721A_9...,5W26153559,EC80ESP,BNDWS,11,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187327,1248454,123904424,2020-03-06 14:00:26,Low (Severity Low) Catalyst Tank Level,NaN,04384413*22383729*082218154102*60701732*G1*BGT*,80156139,6X1u17D1500000000,CMMNS,0,...,NaN,NaN,NaN,NaN,NaN,NaN,False,Severity Low,1.0,False
1187328,1248455,123905139,2020-03-06 14:04:23,Condition Exists Engine Protection Torque Derate,NaN,04358814*06099720*030816202706*09400153*G1*BDR*,79932020,6X1u13D1500000000,CMMNS,0,...,NaN,NaN,65.01096,NaN,73.2,7.83,False,NaN,4.0,False
1187329,1248456,123905996,2020-03-06 14:13:38,Abnormal Rate of Change Aftertreatment 1 Intak...,NaN,05317106*05100987*050719120655*09401585*G1*BDR*,79880653,6X1u13D1500000000,CMMNS,0,...,NaN,NaN,66.57410,NaN,100.0,6.96,False,NaN,NaN,False
1187330,1248457,123906113,2020-03-06 14:14:13,Low (Severity Medium) Engine Coolant Level,NaN,04384413*22544852*090619141107*60701756*G1*BGT*,NaN,NaN,NaN,0,...,NaN,NaN,11.84489,14.1,100.0,1.74,False,Severity Medium,2.0,False


### filter out false derates

In [57]:
combined_filtered = combined_filtered[~((combined_filtered['spn'] == 5246) & (combined_filtered['IsDerateActual'] == False))]

In [58]:
# datetime format
combined_filtered['EventTimeStamp'] = pd.to_datetime(combined_filtered['EventTimeStamp'])

# sort
combined_filtered = combined_filtered.sort_values(['EquipmentID', 'EventTimeStamp'])

# intitial target column
combined_filtered['DeratePredictionTarget'] = 0

# dataframe with just the derate events
derate_events = combined_filtered[combined_filtered['IsDerateActual']].copy()

# group by EquipmentID 
for equipment_id, group in combined_filtered.groupby('EquipmentID'):
    # Get derate events for this truck only
    truck_derates = derate_events[derate_events['EquipmentID'] == equipment_id]
    
    if len(truck_derates) > 0:
        # get indices and timestamps for this truck's rows
        truck_indices = group.index
        truck_timestamps = group['EventTimeStamp'].values
        
        # for each derate event in this truck
        for _, derate_row in truck_derates.iterrows():
            derate_time = derate_row['EventTimeStamp']
            
            # window: 
            window_start = derate_time - pd.Timedelta(hours=8)
            window_end = derate_time - pd.Timedelta(hours=.001)
            
            # all events in the prediction window
            in_window = (truck_timestamps >= window_start) & (truck_timestamps <= window_end)
            indices_to_mark = truck_indices[in_window]
            
            # marked as predicting a derate
            combined_filtered.loc[indices_to_mark, 'DeratePredictionTarget'] = 1

# results
print(f"Total events: {len(combined_filtered)}")
print(f"Events predicting a derate: {combined_filtered['DeratePredictionTarget'].sum()}")

/var/folders/h8/51mqc9v517x00_mhcb5s7ncw0000gn/T/ipykernel_15672/3590571110.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_filtered['EventTimeStamp'] = pd.to_datetime(combined_filtered['EventTimeStamp'])


Total events: 1055387
Events predicting a derate: 1090


## time cut off training(before 2019) and test (after 2018)

In [59]:
cutoff_date = '2018-12-31 23:59:59'
training_faults_before_2019 = combined_filtered[combined_filtered['EventTimeStamp'] <= cutoff_date]
test_faults_after_2019 = combined_filtered[combined_filtered['EventTimeStamp'] > cutoff_date]

In [60]:
training_faults_before_2019.shape

(944122, 51)

In [32]:
#test_faults_after_2019.to_csv('test.csv', index=False)
#training_faults_before_2019.to_csv('training.csv', index=False)

i need to make the training and test data into csv files